# Data Mining Assignment - Adult Dataset Analysis

This notebook implements all 12 required data mining tasks using the UCI Adult dataset.

## Setup: Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import difflib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.naive_bayes import GaussianNB
from mlxtend.frequent_patterns import apriori, association_rules

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

DATA_PATH = 'Dataset/adult.csv'

## Helper Functions

In [ ]:
def _col_name_map(df):
    """Map normalized column names to actual column names."""
    m = {}
    for c in df.columns:
        key = ''.join([ch if ch.isalnum() else '_' for ch in c.lower()]).strip('_')
        m[key] = c
    return m


def _find_col(df, name):
    """Find a column by normalized name (handles hyphens, underscores, spaces)."""
    key = ''.join([ch if ch.isalnum() else '_' for ch in name.lower()]).strip('_')
    m = _col_name_map(df)
    if key in m:
        return m[key]
    if name in df.columns:
        return name
    candidates = difflib.get_close_matches(key, list(m.keys()), n=1, cutoff=0.6)
    if candidates:
        chosen = m[candidates[0]]
        print(f"Warning: using column '{chosen}' for requested '{name}' (fuzzy match)")
        return chosen
    raise KeyError(f"Column matching '{name}' not found in dataframe")


def print_eval(y_true, y_pred, y_prob=None):
    """Print confusion matrix and evaluation metrics."""
    cm = confusion_matrix(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob) if y_prob is not None else float('nan')
    print("Confusion matrix:")
    print(cm)
    print(f"Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}, ROC-AUC: {auc:.4f}")
    return cm, f1, prec, rec, auc

## 1. Load Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
print(f"\nColumn names:\n{df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

## 2. Error Detection & Fixing

In [ ]:
df_clean = df.copy()
df_clean.replace('?', np.nan, inplace=True)

missing = df_clean.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0])

before = len(df_clean)
df_clean.dropna(inplace=True)
after = len(df_clean)
print(f"\nDropped {before - after} rows with missing values.")
print(f"Remaining rows: {after}")

## 3. Outlier Detection

In [ ]:
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
outlier_idx = set()

print("Outliers detected (Z-score > 3.0):")
for c in numeric_cols:
    col = df_clean[c].astype(float)
    z = (col - col.mean()) / col.std(ddof=0)
    idx = df_clean.index[np.abs(z) > 3.0].tolist()
    if idx:
        print(f"  {c}: {len(idx)} outliers")
        outlier_idx.update(idx)

print(f"\nTotal rows with outlier values: {len(outlier_idx)}")

## 4. Linear Regression (Single Variable)

In [ ]:
age_col = _find_col(df_clean, 'age')
hours_col = _find_col(df_clean, 'hours-per-week')

X = df_clean[[age_col]].values
y = df_clean[hours_col].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

lr_single = LinearRegression()
lr_single.fit(X_train, y_train)
score = lr_single.score(X_test, y_test)

print(f"Linear Regression R² (hours-per-week ~ age): {score:.4f}")

# Visualization
plt.figure(figsize=(10, 6))
plt.scatter(X_test, y_test, alpha=0.5, label='Actual')
plt.plot(X_test, lr_single.predict(X_test), color='red', linewidth=2, label='Predicted')
plt.xlabel('Age')
plt.ylabel('Hours per week')
plt.title('Linear Regression: Hours per week ~ Age')
plt.legend()
plt.show()

## 5. Multiple Linear Regression

In [ ]:
feat_names = ['age', 'education-num', 'capital-gain', 'capital-loss']
features = [_find_col(df_clean, n) for n in feat_names]

X = df_clean[features].astype(float)
y = df_clean[_find_col(df_clean, 'hours-per-week')].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1)

lr_multi = LinearRegression()
lr_multi.fit(X_train, y_train)
score = lr_multi.score(X_test, y_test)

print(f"Multiple Linear Regression R²: {score:.4f}")
print(f"Coefficients: {dict(zip(features, lr_multi.coef_))}")

## 6. Polynomial Regression

In [ ]:
age_col = _find_col(df_clean, 'age')
hours_col = _find_col(df_clean, 'hours-per-week')

X = df_clean[[age_col]].values
y = df_clean[hours_col].values

pf = PolynomialFeatures(degree=2, include_bias=False)
Xp = pf.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(Xp, y, test_size=0.25, random_state=0)

lr_poly = LinearRegression()
lr_poly.fit(X_train, y_train)
score = lr_poly.score(X_test, y_test)

print(f"Polynomial Regression (degree 2) R²: {score:.4f}")

## 7. Logistic Regression (Income Classification)

In [ ]:
df_class = df_clean.copy()
income_col = _find_col(df_class, 'income')
df_class[income_col] = df_class[income_col].str.strip()
y = (df_class[income_col] == '>50K').astype(int)

num_names = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
num_cols = [_find_col(df_class, n) for n in num_names]
X = df_class[num_cols].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr_clf = LogisticRegression(max_iter=1000)
lr_clf.fit(X_train_s, y_train)
preds = lr_clf.predict(X_test_s)
probs = lr_clf.predict_proba(X_test_s)[:, 1]

print("Logistic Regression Results:")
print_eval(y_test, preds, probs)

## 8. Decision Tree Classifier

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=2)

dt = DecisionTreeClassifier(random_state=0)
dt.fit(X_train, y_train)
preds = dt.predict(X_test)
probs = dt.predict_proba(X_test)[:, 1]

print("Decision Tree Results:")
cm, f1, prec, rec, auc = print_eval(y_test, preds, probs)

# Confusion Matrix Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['≤50K', '>50K'], yticklabels=['≤50K', '>50K'])
plt.title('Confusion Matrix - Decision Tree')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.show()

## 9. K-Nearest Neighbors Classifier

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=3)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_s, y_train)
preds = knn.predict(X_test_s)
probs = knn.predict_proba(X_test_s)[:, 1]

print("K-Nearest Neighbors (k=5) Results:")
print_eval(y_test, preds, probs)

## 10. K-Means Clustering

In [ ]:
cols = [_find_col(df_clean, n) for n in ['age', 'education-num', 'hours-per-week']]
X_cluster = df_clean[cols].astype(float)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

km = KMeans(n_clusters=3, random_state=0)
labels = km.fit_predict(X_scaled)

print(f"KMeans Inertia: {km.inertia_:.2f}")
print(f"\nCluster distribution:")
print(pd.Series(labels).value_counts().sort_index())

## 11. Naive Bayes Classification

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=4)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

nb = GaussianNB()
nb.fit(X_train_s, y_train)
preds = nb.predict(X_test_s)
probs = nb.predict_proba(X_test_s)[:, 1]

print("Naive Bayes Classification Results:")
print_eval(y_test, preds, probs)

## 12. Apriori Algorithm (Association Rules)

In [ ]:
cats = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex']
actual = []
for c in cats:
    try:
        actual.append(_find_col(df_clean, c))
    except KeyError:
        continue

if not actual:
    actual = df_clean.select_dtypes(include=[object]).columns.tolist()

dfc = df_clean[actual].apply(lambda x: x.str.strip())
df_ohe = pd.get_dummies(dfc)

frequent = apriori(df_ohe, min_support=0.05, use_colnames=True)
rules = association_rules(frequent, metric='confidence', min_threshold=0.6)

print(f"Found {len(frequent)} frequent itemsets")
print(f"Found {len(rules)} association rules")

if len(rules) > 0:
    print("\nTop 5 rules (by lift):")
    print(rules.sort_values('lift', ascending=False)[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head())

## Summary

This notebook has demonstrated all 12 required data mining tasks:

1. ✓ Error detection and fixing (missing values)
2. ✓ Outlier detection (Z-score based)
3. ✓ Linear regression (single variable)
4. ✓ Multiple linear regression
5. ✓ Polynomial regression
6. ✓ Logistic regression (income classification)
7. ✓ Decision tree classifier
8. ✓ Apriori algorithm (association rules)
9. ✓ Evaluation metrics (confusion matrix, F1, precision, recall, ROC-AUC)
10. ✓ K-nearest neighbors classifier
11. ✓ K-means clustering
12. ✓ Naive Bayes classification